In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 

In [ ]:
conn = sqlite3.connect(r"../data/inventory.db")

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
for table in tables['name']:
    print(f'Table name: {table}')
    display(pd.read_sql(f"select * from {table} limit 5", conn))

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
for table in tables['name']:
    print(f'Table name: {table}')
    display(pd.read_sql(f"select * from {table} limit 5", conn).dtypes)

In [ ]:
purchase_agg_df = pd.read_sql_query("""
select
p.PONumber,
count(distinct p.Brand) as total_brands,
sum(p.Quantity) as total_item_quantity,
sum(p.Dollars) as total_item_dollars,
avg(julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_receiving_delay
from purchases p
group by p.PONumber
""",conn)

### Aggregate Purchase Data by Purchase Order

This query summarizes line-item records from the `purchases` table into a single row per Purchase Order (`PONumber`) to compute the following metrics:
- **`total_brands`**: Count of distinct brands purchased in the PO.
- **`total_item_quantity`**: Total physical units ordered across all items.
- **`total_item_dollars`**: Total monetary cost of the items.
- **`avg_receiving_delay`**: Average time difference (in days) between the Purchase Order date (`PODate`) and the receiving date (`ReceivingDate`).

In [ ]:
purchase_agg_df.head()

In [ ]:
pd.read_sql_query("""
select
vi.PONumber,
vi.Quantity as invoice_quantity,
vi.Dollars as invoice_dollars,
vi.Freight,
(julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
(julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay
from vendor_invoice vi
""",conn)

### Extract Vendor Invoice Metrics

This query retrieves invoice-level details from the `vendor_invoice` table and computes the duration of key billing lifecycle phases:
- **`invoice_quantity` & `invoice_dollars`**: The total physical quantity and monetary amount billed on the vendor invoice.
- **`Freight`**: The freight cost associated with the invoice.
- **`days_po_to_invoice`**: The time difference (in days) between the Purchase Order date (`PODate`) and the invoice date (`InvoiceDate`).
- **`days_to_pay`**: The time difference (in days) between the invoice date (`InvoiceDate`) and the payment date (`PayDate`).

In [ ]:
query = """
SELECT 
    vi.PONumber,
    vi.Quantity AS invoice_quantity,
    vi.Dollars AS invoice_dollars,
    vi.Freight,
    CAST((julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS INTEGER) AS days_po_to_invoice,
    CAST((julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS INTEGER) AS days_to_pay,
    COUNT(DISTINCT p.Brand) AS total_brands,
    SUM(p.Quantity) AS total_item_quantity,
    SUM(p.Dollars) AS total_item_dollars,
    AVG(julianday(p.ReceivingDate) - julianday(vi.PODate)) AS avg_receiving_delay
FROM vendor_invoice vi
LEFT JOIN purchases p 
    ON vi.PONumber = p.PONumber
GROUP BY 
    vi.PONumber, vi.Quantity, vi.Dollars, vi.Freight, vi.InvoiceDate, vi.PODate, vi.PayDate;
"""

df = pd.read_sql_query(query, conn)

### Combine Vendor Invoices and Purchase Metrics

This query performs a left join between the `vendor_invoice` and `purchases` tables, grouping by invoice details to compile a comprehensive risk-analysis dataset:
- **`days_po_to_invoice` & `days_to_pay`**: Lifecycle durations cast to integers representing days from PO to invoice, and invoice to payment.
- **`total_brands`, `total_item_quantity`, `total_item_dollars`**: Aggregated purchase order characteristics matched directly against the corresponding vendor invoice.
- **`avg_receiving_delay`**: The average time elapsed between the PO date and the item receiving date.

In [ ]:
pd.read_sql_query("""
WITH purchase_agg AS (
    SELECT
        p.PONumber,
        COUNT(DISTINCT p.Brand) AS total_brands,
        SUM(p.Quantity) AS total_item_quantity,
        SUM(p.Dollars) AS total_item_dollars,
        AVG(julianday(p.ReceivingDate) - julianday(p.PODate)) AS avg_receiving_delay
    FROM purchases p
    GROUP BY p.PONumber
)

SELECT
    vi.PONumber,
    vi.Quantity AS invoice_quantity,
    vi.Dollars AS invoice_dollars,
    vi.Freight,
    (julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
    (julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
    pa.total_brands,
    pa.total_item_quantity,
    pa.total_item_dollars,
    pa.avg_receiving_delay
FROM vendor_invoice vi
LEFT JOIN purchase_agg pa 
    ON vi.PONumber = pa.PONumber
""", conn)



### Combine Vendor Invoices and Purchase Metrics via CTE

This query uses a Common Table Expression (`WITH purchase_agg AS (...)`) to pre-aggregate purchase-order-level metrics before performing a left join against the `vendor_invoice` table. It builds the primary dataset for downstream risk modeling by capturing:
- **Invoice Financials & Logistics**: `invoice_quantity`, `invoice_dollars`, and `Freight`.
- **Process Lifecycle Durations**: `days_po_to_invoice` (time from order creation to invoice date) and `days_to_pay` (time from invoice date to payment).
- **Aggregated Purchase Characteristics**: `total_brands`, `total_item_quantity`, `total_item_dollars`, and `avg_receiving_delay` matched cleanly from the purchase order data.

In [ ]:
print("Missing values:")
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
# Export the raw dataframe for preprocessing
df.to_csv("processed_invoices.csv", index=False)
print(f"Dataframe successfully exported with shape: {df.shape}")